In [21]:
%pip install datasets SentencePiece  rouge evaluate nltk  rouge_score

Note: you may need to restart the kernel to use updated packages.


In [22]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset, load_from_disk, Dataset
from evaluate import load

In [23]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [24]:
# dataset = load_dataset("abisee/cnn_dailymail", "1.0.0")
dataset = load_from_disk("CNN-dailymail")

load_recs = 300

train_dataset = dataset['train'].shuffle(seed=42).select(range(load_recs))
test_dataset = dataset['test'].shuffle(seed=12).select(range(30))
# valid_dataset = dataset['validation']

train_dataset

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 300
})

In [25]:
import torch
print(torch.cuda.is_available())  # Ensure CUDA is available
print(torch.cuda.current_device())  # Check the current device
print(torch.__version__)  # Check PyTorch version
print(torch.cuda.get_device_name(0)) 

True
0
2.7.0+cu126
NVIDIA GeForce GTX 1050 Ti


In [26]:
X = tokenizer(
    train_dataset['article'],
    padding=True,
    truncation=True,
    max_length=512
)
X_labels = tokenizer(
    train_dataset['highlights'],
    padding= True,
    truncation= True,
    max_length= 512
)
X['labels'] = X_labels['input_ids']

X1 = tokenizer(
    test_dataset['article'],
    padding=True,
    truncation=True,
    max_length=512
)
X1_labels = tokenizer(
    test_dataset['highlights'],
    padding=True,
    truncation=True,
    max_length=512
)
X1['labels']=X1_labels['input_ids']



In [27]:
X[:5]

{'input_ids': [[938,
   3,
   5,
   11016,
   12528,
   3,
   5,
   3,
   10744,
   8775,
   20619,
   2326,
   10,
   3,
   5,
   10668,
   10,
   4928,
   3,
   6038,
   6,
   204,
   1332,
   2038,
   3,
   5,
   1820,
   3,
   5,
   3,
   6880,
   4296,
   11430,
   10,
   3,
   5,
   12046,
   10,
   4560,
   3,
   6038,
   6,
   204,
   1332,
   2038,
   3,
   5,
   5245,
   724,
   13,
   8,
   337,
   384,
   113,
   3977,
   16,
   3,
   9,
   14491,
   22133,
   45,
   4146,
   1911,
   6778,
   15,
   14566,
   53,
   133,
   43,
   118,
   25429,
   3,
   31,
   4065,
   77,
   676,
   31,
   6,
   16273,
   7,
   243,
   469,
   5,
   37,
   5678,
   13,
   4464,
   1158,
   1079,
   11,
   31423,
   6176,
   130,
   3883,
   5815,
   70,
   3062,
   6,
   7758,
   60,
   35,
   6,
   44,
   8,
   1156,
   234,
   79,
   2471,
   30,
   4691,
   1635,
   109,
   1210,
   1061,
   16,
   5184,
   12940,
   6,
   4653,
   26334,
   5,
   37,
   16,
   10952,
   7,
   43,
   

In [28]:

train_dataset = Dataset.from_dict(X)
test_dataset = Dataset.from_dict(X1)

In [29]:
from transformers import TrainingArguments, Seq2SeqTrainingArguments


# Define the training arguments
training_args = Seq2SeqTrainingArguments(
    # output_dir="./results1",            # Where to save the model checkpoints
    eval_strategy='steps',
    # learning_rate=1e-5,                # Learning rate for training
    per_device_train_batch_size=10,    # Batch size for training
    per_device_eval_batch_size=10,     # Batch size for evaluation
    generation_max_length=128,  # or 256 if your summaries are long
    generation_num_beams=2,   
    num_train_epochs=1,
    weight_decay=0.01,
    # weight_decay=0.01,                 # Weight decay for regularization
    # logging_dir="./logs",              # Where to save logs
    # eval_steps=1,
    logging_steps=10,                  # Log every 10 steps
    # disable_tqdm=True,
    predict_with_generate=True,
    # label_names=['labels'],
    load_best_model_at_end=True,       # Load the best model at the end of training
    metric_for_best_model="ROUGE-1"   # Recommended metric for classification tasks
)


In [30]:
from transformers import Trainer, Seq2SeqTrainer
# from sklearn.metrics import accuracy_score
# from evaluate import load
import numpy as np

# accuracy = load("accuracy")

# Load metric
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=1)
#     # return {'accuracy': acc, 'eval_accuracy': acc}
#     return accuracy.compute(predictions=predictions, references=labels)
# Prepare the trainer

import evaluate
import numpy as np

rouge = evaluate.load('rouge')

# Define compute_metrics function to calculate ROUGE scores
def compute_metrics(eval_pred):
    """
     Evaluation output (always contains labels), to be used to compute metrics.

    Parameters:
        predictions (`np.ndarray`): Predictions of the model.
        label_ids (`np.ndarray`): Targets to be matched.
        inputs (`np.ndarray`, *optional*): Input data passed to the model.
        losses (`np.ndarray`, *optional*): Loss values computed during evaluation.
    """
    predictions, labels = eval_pred

    # Replace -100 (used for padding in labels) with pad_token_id so decoding works
    # labels = np.where(labels != -100, tokenizer.pad_token_id, labels)

    # Decode
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    for i in range(len(decoded_preds)):
        print("=================")
        print("Predication:")
        print(decoded_preds[i])
        print("Expected:")
        print(decoded_labels[i])
    print("prediction:")
    print(decoded_preds, flush=True)
    print("expected:")
    print(decoded_labels, flush=True)
    # print("Actual Inputs:")
    # print(inputs,flush=True)
    # Optional: Strip whitespace to avoid formatting mismatches
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Extract ROUGE-L, ROUGE-1, ROUGE-2 (F1 scores)
    # result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    
    # Optional: round for easier reading
    result = {k: round(v, 4) for k, v in result.items()}
    print(result, flush=True)
    result['eval_ROUGE-1'] = result['rouge1']
    return result


trainer = Seq2SeqTrainer(
    model=model,   
    args=training_args,                    # Training arguments defined earlier
    train_dataset=train_dataset,   # Training dataset
    eval_dataset=test_dataset,     # Evaluation dataset
    tokenizer=tokenizer,   
    compute_metrics=compute_metrics  # Accuracy metric

)

trainer.train()

/tmp/ipykernel_7784/2813581052.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss,Validation Loss,Rouge-1,Rouge1,Rouge2,Rougel,Rougelsum
10,11.206300,7.214479,0.388700,0.388700,0.180800,0.288900,0.287000
20,7.431300,5.446099,0.370500,0.370500,0.165700,0.274800,0.273500
30,5.436500,4.915690,0.365900,0.365900,0.160500,0.267800,0.266300


Predication:
in a remote Amazon village in Venezuela who had not had previous contact with non-Yanomami. this information revealed, for the first time, the species of bacteria that co-exist with people who have never been exposed to industrialized society. researchers have completed the first comprehensive study of the microbes living on and in and uncontacted tribe from the Amazon. they say the results show just how modern lifestyles and lifestyles have changed us - and that the bacteria they found could be potentially beneficial to modern society.
Expected:
Researchers sequenced microbiomes of Yanomami people in Amazon. Testing found they harbour microbiomes with the highest diversity of bacteria and genetic functions ever reported in a human group. Bacteria they found could be potentially beneficial to modern society.
Predication:
Ben Stokes gave marlon Samuels a salute on his way to the second Test. 'I don’t think Marlon meant anything by it. You have to find ways to relax and have

TrainOutput(global_step=30, training_loss=8.024680201212565, metrics={'train_runtime': 251.8363, 'train_samples_per_second': 1.191, 'train_steps_per_second': 0.119, 'total_flos': 40602540441600.0, 'train_loss': 8.024680201212565, 'epoch': 1.0})